# 第6章 用 rocprof 找到慢在哪里

**操作手册** | 对照两个 vector add，只看 kernel 时间、工作划分和 stride 趋势

本手册对应文档：`docs/part1-profiling/chapter6/index.md`  
本手册对应代码：`code/part1-profiling/chapter6/`

---

### 本章导读

> 本章只解决一个问题：**两个 vector add 实现速度差很多时，怎么先找到慢在哪个 kernel？**
>
> 我们会先用 benchmark 看差距，再用 `rocprofv3` 查看每次 kernel dispatch，最后扫描 `stride` 观察变化趋势。这个案例还会提醒你：命令行里只改一个参数，不代表 GPU 内部只改了一件事。读完后，你应该会定位慢点，也知道何时还需要更多证据，再下结论。

本章代码在 `code/part1-profiling/chapter6/vector_add.hip`。下面的性能数据来自 **Radeon RX 9070 XT（gfx1201）+ ROCm 7.13 + 原生 Ubuntu 24.04**；换一张卡，数字会变，但操作顺序不变。

## 在云端运行本章
本教程以 RX 9070 XT（`gfx1201` / RDNA4）为讲解和参考平台。云端运行时，请用当前平台的数据判断代码是否正确、趋势是否一致；不同 GPU 的绝对性能不宜直接比较。

云平台已预装 ROCm、PyTorch 和基础编译工具，可跳过本地 `uv` 与激活步骤。本地读者仍按原环境步骤操作；`rocprofv3` 等额外工具的可用性会在本章单独说明。

编译架构以当前 `rocminfo` 输出为准；未识别时先检查环境。


> 本教程仓库的本地 uv 环境现统一为 **ROCm 10.0**，安装方法见[第 1 章](../../docs/part0-intro/chapter1/index.md)。本 Notebook 引用的历史实验保留采集时的软件版本；云平台预装环境以当前运行时检测结果为准。


## Goal

学会用 `rocprofv3` 定位慢 kernel，理解工作划分对性能的影响。具体目标：

1. 用 benchmark 确认两个实现的速度差距
2. 用 `rocprofv3 --kernel-trace` 找到慢在哪个 dispatch
3. 对比 coalesced 和 linecross 的 Grid Size、VGPR、SGPR
4. 扫描 stride 参数，观察性能趋势
5. 识别实验同时改变了哪些变量（地址排布、循环次数、Grid Size）

## Prerequisite

- 云端预装 ROCm/PyTorch；跳过本地 `uv` / `activate` 段
- `rocprofv3` 可选（仅用于 kernel trace；工具或权限不可用时跳过）
- 理解 warmup、repeat、GPU event 计时

## 平台

本手册基于以下环境验证：

- **GPU**: AMD Radeon RX 9070 XT (gfx1201)
- **历史参考数据的 ROCm**: 7.13
- **OS**: Ubuntu 24.04 (native, kernel 6.17.0-35-generic)
- **历史参考数据的 hipcc**: 7.13.99004 / arch gfx1201

其他 RDNA3/RDNA4 架构（gfx1100, gfx1151, gfx1201）均可运行，数字会有差异。

## 6.1 先看懂两个实现

这一节先看两个 kernel 分别怎样把 `n` 个元素分给线程。它们的输出相同，但线程的工作划分并不相同。

### 6.1.1 连续访存版

普通 vector add 让线程 `i` 处理元素 `i`：

```cpp
__global__ void kernel_coalesced(const float* a, const float* b,
                                 float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}
```

一个 wavefront 里的 lane 0、lane 1、lane 2 会依次访问 `a[0]`、`a[1]`、`a[2]`。这些地址连在一起，GPU 可以把多条线程请求合并成较少的内存事务。这就是**合并访存（Memory Coalescing）**。

这个版本每个线程只计算一个输出，因此一共启动约 `n` 个线程。

### 6.1.2 linecross 版

`linecross` 是本章给对照实现起的名字，不是 ROCm 的标准术语。它让每个 lane 处理一小段连续元素：

```cpp
int tile_base = wave_id * 32 * stride;
int my_start = tile_base + lane * stride;
for (int j = 0; j < stride; ++j) {
    int i = my_start + j;
    c[i] = a[i] + b[i];
}
```

以 `stride=32` 为例，同一次循环中，各 lane 看到的地址大致是：

```text
lane 0  -> a[0]
lane 1  -> a[32]
lane 2  -> a[64]
lane 3  -> a[96]
...
```

相邻 lane 的起点隔了 32 个 float，也就是 128 字节。地址排布比连续访存版更分散。

一个 wavefront 一共处理 `32 × stride` 个输出，所以 stride 变大时会同时发生两件事：

1. 每个 lane 的循环次数增加；
2. 需要启动的 wavefront 数减少。

*（图示：两个实现同时改变了地址排布和线程工作划分。连续访存版——lane 地址相邻、每个线程 1 个输出、启动约 n 个线程；linecross stride=32——lane 起点相隔 128 字节、每个线程 32 个输出、启动约 n/32 个线程。）*

这不是一个“只改地址排布”的严格对照。它适合练习 benchmark 和 kernel trace，也能展示 stride 增大时的整体趋势；这组数据不足以把全部性能差距都归因于访存合并。

## Parameter

### vector_add_bench 参数

| 参数 | 含义 | 示例值 |
|------|------|--------|
| `--kernel` | 选择 coalesced 或 linecross | coalesced |
| `--size` | 处理元素个数 | 16777216 (16M) |
| `--block` | 每个 block 的线程数 | 256 |
| `--stride` | linecross 中每个 lane 负责的连续元素数 | 1, 32 |
| `--warmup` | 热身次数 | 20 |
| `--repeat` | 正式计时次数 | 100 |
| `--output-json` | 保存结果到 JSON | logs/result.json |

### 有效带宽的计算

程序会同时输出延迟和**有效带宽**。按算法口径，vector add 每个元素需要读 `a`、读 `b`、写 `c`，合计 12 B 有效数据，因此：

```text
有效带宽 = 12 × 元素个数 / kernel 时间
```

有效带宽是为了方便比较而换算出的数值，不等于硬件实际发出的 DRAM 事务量。

### rocprofv3 参数（可选 capability）

| 参数 | 含义 |
|------|------|
| `--kernel-trace` | 收集 kernel dispatch 跟踪 |
| `--output-directory <dir>` | 为本次运行建立独立输出目录 |
| `--output-file <label>` | 为 trace 文件设置 label |
| `--output-format csv` | 输出格式为 CSV |

trace 只有在命令成功且生成非空 CSV 时才算可用。

## Execution

### 步骤1：定位仓库根目录并进入工作目录

In [ ]:
import csv
import json
import os
import re
import secrets
import statistics
import subprocess
import sys
from pathlib import Path

def find_repo_root():
    """Find the repository by markers, independent of the current path text."""
    starts = []
    notebook_file = globals().get("__file__")
    if notebook_file:
        starts.append(Path(notebook_file).resolve())
    starts.append(Path.cwd().resolve())
    visited = set()
    for start in starts:
        current = start if start.is_dir() else start.parent
        for candidate in (current, *current.parents):
            if candidate in visited:
                continue
            visited.add(candidate)
            if all((candidate / marker).is_dir() for marker in ("code", "docs", "notebooks")):
                return candidate
    raise RuntimeError("无法从当前 Notebook 路径找到包含 code/docs/notebooks 的仓库根目录")

def run_capture(cmd, *, timeout, cwd=None):
    """Run one external command and always expose returncode/stderr."""
    try:
        result = subprocess.run(
            cmd,
            cwd=cwd,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
    except FileNotFoundError as exc:
        print(f"command unavailable: {cmd[0]} | returncode=NOT_FOUND | stderr={exc}")
        return None
    except subprocess.TimeoutExpired as exc:
        stderr = exc.stderr or ""
        if isinstance(stderr, bytes):
            stderr = stderr.decode(errors="replace")
        print(f"command timeout after {timeout}s: {' '.join(map(str, cmd))} | returncode=TIMEOUT")
        print(f"stderr={stderr.strip() or '<empty>'}")
        return None
    print(f"returncode={result.returncode}")
    print(f"stderr={result.stderr.strip() or '<empty>'}")
    return result

REPO_ROOT = find_repo_root()
WORK_DIR = REPO_ROOT / "code/part1-profiling/chapter6"
if not WORK_DIR.is_dir():
    raise RuntimeError(f"缺少工作目录: {WORK_DIR}")
os.chdir(WORK_DIR)

# Every fresh execution starts with no architecture or compile capability.
ARCH_READY = False
GPU_ARCH = None
ARCH_SOURCE = "unavailable"
arch_source = "unavailable"
COMPILE_OK = False
RUN_READY = False
COMPILE_TOKEN = None
rocprof_available = False
PROFILE_STATUS = "NOT_RUN"
CH6_PROFILE_COMPLETE = False
stride_scan_status = "NOT_RUN"

print(f"仓库根目录: {REPO_ROOT}")
print(f"工作目录: {WORK_DIR}")
print(f"当前目录: {Path.cwd()}")

### 步骤2：检测 GPU 架构

编译前检测当前 GPU 架构：可用 `HELLO_GPU_ARCH` 显式指定 `gfx1100`、`gfx1151` 或 `gfx1201` 作为仅编译 target；否则只解析带 timeout 的 `rocminfo` 中唯一精确 GPU Agent `Name: gfx...`，并在可用时打印 wavefront size。

In [ ]:
# 检测 GPU 架构：override 只控制 hipcc target，不设置任何运行时 HSA override。
def detect_gpu_arch():
    """Return (arch, source), or stop on invalid/ambiguous detection."""
    override = os.environ.get("HELLO_GPU_ARCH", "").strip()
    if override:
        if override not in {"gfx1100", "gfx1151", "gfx1201"}:
            raise RuntimeError(
                f"HELLO_GPU_ARCH 无效: {override!r}；仅接受 gfx1100/gfx1151/gfx1201"
            )
        return override, "HELLO_GPU_ARCH (compile target; not hardware proof)"

    result = run_capture(["rocminfo"], timeout=10)
    if result is None:
        raise RuntimeError("rocminfo 不可用或超时，无法确认 GPU arch")
    if result.returncode != 0:
        raise RuntimeError(
            f"rocminfo 失败（returncode={result.returncode}）: "
            f"{result.stderr.strip() or '<empty>'}"
        )

    names = []
    wavefront_sizes = []
    for line in result.stdout.splitlines():
        match = re.fullmatch(r"\s*Name:\s*(gfx[0-9a-z]+)\s*", line)
        if match:
            names.append(match.group(1))
        wave_match = re.fullmatch(r"\s*Wavefront Size:\s*(\d+)\s*", line)
        if wave_match:
            wavefront_sizes.append(wave_match.group(1))
    if len(names) != 1:
        candidates = sorted(set(names))
        raise RuntimeError(
            "rocminfo 必须给出唯一 GPU Agent Name: gfx...；"
            f"实际匹配 {len(names)} 个: {candidates or '<none>'}"
        )
    if wavefront_sizes:
        print(f"rocminfo wavefront size: {', '.join(sorted(set(wavefront_sizes)))}")
    return names[0], "rocminfo"

# Re-running this cell invalidates any previous binary/token before compilation.
ARCH_READY = False
GPU_ARCH = None
ARCH_SOURCE = "unavailable"
arch_source = "unavailable"
COMPILE_OK = False
RUN_READY = False
COMPILE_TOKEN = None

GPU_ARCH, ARCH_SOURCE = detect_gpu_arch()
arch_source = ARCH_SOURCE
ARCH_READY = True
print(f"arch={GPU_ARCH} arch_source={arch_source}")


## 6.2 先跑一遍，确认谁更慢

这一节先不打开 profiler，只用第 5 章的计时方法比较三个配置。从仓库根目录进入本篇环境并编译，然后运行连续访存版，再把 `linecross` 的 stride 分别设为 1 和 32。

### 步骤3：编译 vector_add_bench

编译 HIP 程序，生成 benchmark 可执行文件：

In [ ]:
logs_dir = WORK_DIR / "logs"
logs_dir.mkdir(exist_ok=True)

# The target name carries the selected arch; a prior binary can never satisfy this run.
source_file = WORK_DIR / "vector_add.hip"
output_binary = WORK_DIR / f"vector_add_bench_{GPU_ARCH or 'unavailable'}"
COMPILE_OK = False
RUN_READY = False
COMPILE_TOKEN = None

if not ARCH_READY or not GPU_ARCH:
    print("编译阻止: GPU arch 未成功检测；不会使用任何旧 binary")
else:
    try:
        output_binary.unlink(missing_ok=True)
        print(f"已删除本轮目标的旧 binary: {output_binary.name}")
    except OSError as exc:
        print(f"编译失败: 无法隔离旧 binary {output_binary}: {exc}")
    else:
        if not source_file.is_file():
            print(f"编译失败: 未找到源文件 {source_file}")
        else:
            compile_cmd = [
                "hipcc",
                f"--offload-arch={GPU_ARCH}",
                "-O3",
                str(source_file),
                "-o",
                str(output_binary),
            ]
            print(f"编译命令: {' '.join(compile_cmd)}")
            result = run_capture(compile_cmd, timeout=120, cwd=WORK_DIR)
            if result is not None and result.returncode == 0 and output_binary.is_file():
                COMPILE_OK = True
                COMPILE_TOKEN = f"{GPU_ARCH}:{secrets.token_hex(8)}"
                RUN_READY = True
                print(f"编译成功: {output_binary}")
                print(f"compile_token={COMPILE_TOKEN}")
            else:
                returncode = "UNAVAILABLE" if result is None else result.returncode
                stderr = "<see command diagnostics>" if result is None else (result.stderr.strip() or "<empty>")
                print(f"编译失败: returncode={returncode} stderr={stderr}")
                try:
                    output_binary.unlink(missing_ok=True)
                except OSError as cleanup_exc:
                    print(f"编译失败后无法删除 partial binary: {cleanup_exc}")
                print("后续 benchmark/profile/stride 全部阻止，不消费旧 binary 或旧结果")

### 步骤4：运行 benchmark — coalesced 版本

先用本轮成功编译的 binary 运行连续访存版；只有新鲜 JSON 才会被接受。

### 步骤5：运行 benchmark — linecross stride=1

`linecross stride=1` 每个线程也只处理一个元素，线程数和连续访存版相同，因此两者时间应该接近。

In [ ]:
BENCHMARK_RESULTS = {}

def run_benchmark(label, kernel_args):
    """Run only this compile's binary and consume only a fresh JSON result."""
    if not (RUN_READY and COMPILE_OK and COMPILE_TOKEN and output_binary.is_file()):
        print(f"{label}: BLOCKED (compile_token 不可用，不使用旧 binary/JSON)")
        BENCHMARK_RESULTS[label] = False
        return False
    output_json = logs_dir / f"{label}.json"
    try:
        output_json.unlink(missing_ok=True)
    except OSError as exc:
        print(f"{label}: FAILED before launch; cannot remove old JSON: {exc}")
        BENCHMARK_RESULTS[label] = False
        return False
    cmd = [str(output_binary), *kernel_args, "--output-json", str(output_json)]
    print(f"运行 {label}: {' '.join(cmd)}")
    result = run_capture(cmd, timeout=300, cwd=WORK_DIR)
    if result is None or result.returncode != 0:
        returncode = "UNAVAILABLE" if result is None else result.returncode
        stderr = "<see command diagnostics>" if result is None else (result.stderr.strip() or "<empty>")
        print(f"{label}: FAILED returncode={returncode} stderr={stderr}")
        BENCHMARK_RESULTS[label] = False
        return False
    print(result.stdout)
    if not output_json.is_file() or output_json.stat().st_size == 0:
        print(f"{label}: FAILED; fresh JSON was not produced")
        BENCHMARK_RESULTS[label] = False
        return False
    try:
        json.loads(output_json.read_text(encoding="utf-8"))
    except (OSError, ValueError) as exc:
        print(f"{label}: FAILED; invalid fresh JSON: {exc}")
        BENCHMARK_RESULTS[label] = False
        return False
    print(f"{label}: PASS; fresh JSON={output_json}")
    BENCHMARK_RESULTS[label] = True
    return True

coalesced_ok = run_benchmark(
    "coalesced_size16777216",
    ["--kernel", "coalesced", "--size", "16777216", "--block", "256",
     "--warmup", "20", "--repeat", "100"],
)
linecross1_ok = run_benchmark(
    "linecross_stride1_size16777216",
    ["--kernel", "linecross", "--size", "16777216", "--block", "256",
     "--stride", "1", "--warmup", "20", "--repeat", "100"],
)

### 步骤6：运行 benchmark — linecross stride=32

到了 `stride=32`，每个线程处理 32 个元素，Grid Size 也会缩小到原来的 1/32。

In [ ]:
linecross32_ok = run_benchmark(
    "linecross_stride32_size16777216",
    ["--kernel", "linecross", "--size", "16777216", "--block", "256",
     "--stride", "32", "--warmup", "20", "--repeat", "100"],
)

### benchmark 结果解读

在 9070XT 上得到的参考结果如下：

| kernel | stride | 最短时间 | 有效带宽 | 正确性 |
|--------|--------|----------|----------|--------|
| coalesced | - | 0.334 ms | 603 GB/s | OK |
| linecross | 1 | 0.336 ms | 599 GB/s | OK |
| linecross | 32 | 2.25 ms | 89.7 GB/s | OK |

`linecross stride=1` 每个线程也只处理一个元素，线程数和连续访存版相同，因此两者时间接近。

到了 `stride=32`，时间增加到 2.25 ms，约为连续访存版的 **6.7 倍**。现在可以确认这个配置更慢，但还需要进一步区分地址分散、wavefront 变少，还是两者共同造成。

## 6.3 用 rocprof 看每次 kernel dispatch

这一节只用 `rocprofv3` 的 kernel trace（核函数跟踪），不碰复杂计数器。GPU event 已经给出了计时结果，kernel trace 的新增价值是把每次 dispatch 单独列出来；以后面对包含很多 kernel 的程序，就能用它找到最慢的那一个。

程序一共启动 15 次 kernel：前 5 次是 warmup，后 10 次才是正式结果。第一次打开生成的 CSV，先找下面几组列：

| 列 | 先用它回答什么 |
|------|------|
| `Kernel_Name` | 到底运行了哪个 kernel |
| `Start_Timestamp` / `End_Timestamp` | 单次 kernel 花了多久 |
| `Grid_Size` | 一共启动了多少个 work-item |
| `VGPR_Count` / `SGPR_Count` | kernel 的寄存器分配 |

时间戳单位是纳秒：

```text
kernel 时间（μs）= (End_Timestamp - Start_Timestamp) / 1000
```

### 步骤7：检查 rocprofv3 可用性

在采集 kernel trace 前，先检查 `rocprofv3` 是否可用：

In [ ]:
rocprof_available = False
PROFILE_STATUS = "UNAVAILABLE"
result = run_capture(["rocprofv3", "--version"], timeout=120, cwd=WORK_DIR)
if result is not None and result.returncode == 0:
    print("rocprofv3 可用:")
    print(result.stdout)
    rocprof_available = True
    PROFILE_STATUS = "READY"
else:
    returncode = "UNAVAILABLE" if result is None else result.returncode
    stderr = "<see command diagnostics>" if result is None else (result.stderr.strip() or "<empty>")
    print(f"rocprofv3 可选能力不可用: returncode={returncode} stderr={stderr}")
    print("跳过 kernel trace；使用内置参考数据继续阅读（不替代本机 profile）")

### 步骤7（续）：采集 kernel trace（如果 rocprofv3 可用）

使用 `rocprofv3 --kernel-trace` 收集每次 kernel dispatch 的时间戳和配置：

In [ ]:
profile_results = {}
profile_traces = {}

def _trace_column(fieldnames, *names):
    """Resolve rocprof CSV column variants without assuming one schema."""
    normalized = {re.sub(r"[^a-z0-9]", "", name.lower()): name for name in fieldnames}
    for name in names:
        match = normalized.get(re.sub(r"[^a-z0-9]", "", name.lower()))
        if match:
            return match
    return None

def _benchmark_min_ms(label):
    path = logs_dir / f"{label}.json"
    try:
        return float(json.loads(path.read_text(encoding="utf-8"))["min_ms"])
    except (KeyError, OSError, TypeError, ValueError) as exc:
        print(f"{label}: benchmark 对照不可用 ({exc})")
        return None

def parse_trace_csv(trace_path, benchmark_label):
    """Parse every kernel row in one non-empty rocprof kernel-trace CSV."""
    try:
        with trace_path.open(newline="", encoding="utf-8-sig") as handle:
            reader = csv.DictReader(handle)
            fieldnames = reader.fieldnames or []
            columns = {
                "kernel": _trace_column(fieldnames, "Kernel_Name", "Kernel Name"),
                "start": _trace_column(fieldnames, "Start_Timestamp", "Start Timestamp"),
                "end": _trace_column(fieldnames, "End_Timestamp", "End Timestamp"),
                "grid": _trace_column(fieldnames, "Grid_Size", "Grid Size"),
                "vgpr": _trace_column(fieldnames, "VGPR_Count", "VGPR Count"),
                "sgpr": _trace_column(fieldnames, "SGPR_Count", "SGPR Count"),
            }
            display_columns = {name: column or "unavailable" for name, column in columns.items()}
            print(f"{trace_path.name}: detected columns={display_columns}")
            if not all(columns[name] for name in ("kernel", "start", "end")):
                print(f"{trace_path.name}: 解析失败；缺少 Kernel_Name/Start/End")
                return False
            kernels = {}
            for row in reader:
                kernel_name = (row.get(columns["kernel"]) or "").strip()
                if not kernel_name:
                    continue
                try:
                    duration_us = (float(row[columns["end"]]) - float(row[columns["start"]])) / 1000.0
                except (KeyError, TypeError, ValueError):
                    print(f"{trace_path.name}: skip {kernel_name}; invalid Start/End")
                    continue
                kernels.setdefault(kernel_name, []).append((duration_us, row))
    except (csv.Error, OSError, UnicodeError) as exc:
        print(f"{trace_path.name}: 解析失败 ({exc})")
        return False

    if not kernels:
        print(f"{trace_path.name}: 解析失败；没有包含有效时间戳的 kernel 行")
        return False
    benchmark_ms = _benchmark_min_ms(benchmark_label)
    for kernel_name, dispatches in kernels.items():
        durations = [duration for duration, _ in dispatches]
        sample = dispatches[0][1]
        def value_or_unavailable(name):
            column = columns[name]
            return sample.get(column, "").strip() if column and sample.get(column, "").strip() else "unavailable"
        trace_median_us = statistics.median(durations)
        comparison = "unavailable" if benchmark_ms is None else f"benchmark min={benchmark_ms * 1000:.3f} us"
        print(
            f"{trace_path.name} | Kernel_Name={kernel_name} | dispatches={len(durations)} | "
            f"Start={value_or_unavailable('start')} | End={value_or_unavailable('end')} | "
            f"min={min(durations):.3f} us | median={trace_median_us:.3f} us | {comparison} | "
            f"Grid={value_or_unavailable('grid')} | VGPR={value_or_unavailable('vgpr')} | "
            f"SGPR={value_or_unavailable('sgpr')}"
        )
    return True

def run_profile(label, benchmark_label, kernel_args):
    """Collect and parse every fresh, non-empty kernel trace CSV for one benchmark."""
    global rocprof_available, PROFILE_STATUS
    if not (rocprof_available and RUN_READY and COMPILE_OK and COMPILE_TOKEN and output_binary.is_file()):
        print(f"{label}: 当前条件不满足，跳过 profile（工具或编译能力不可用）")
        profile_results[label] = False
        return False
    profile_root = logs_dir / "rocprof" / f"{GPU_ARCH}_{COMPILE_TOKEN.replace(':', '_')}"
    output_dir = profile_root / label
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        for old_file in output_dir.iterdir():
            if old_file.is_file():
                old_file.unlink()
    except OSError as exc:
        print(f"{label}: 启动 profile 前失败: {exc}")
        profile_results[label] = False
        PROFILE_STATUS = "FAILED"
        return False
    cmd = [
        "rocprofv3", "--kernel-trace", "--output-directory", str(output_dir),
        "--output-file", label, "--output-format", "csv", "--", str(output_binary), *kernel_args,
    ]
    print(f"采集 {label} kernel trace: {' '.join(cmd)}")
    result = run_capture(cmd, timeout=300, cwd=WORK_DIR)
    if result is None or result.returncode != 0:
        returncode = "UNAVAILABLE" if result is None else result.returncode
        stderr = "<see command diagnostics>" if result is None else (result.stderr.strip() or "<empty>")
        print(f"{label}: profile 失败 returncode={returncode} stderr={stderr}")
        profile_results[label] = False
        PROFILE_STATUS = "FAILED"
        return False
    print(result.stdout)
    try:
        traces = sorted(
            path for path in output_dir.rglob("*.csv")
            if path.is_file() and path.stat().st_size > 0
            and "kernel" in path.name.lower() and "trace" in path.name.lower()
        )
    except OSError as exc:
        print(f"{label}: 检查 trace 时失败: {exc}")
        traces = []
    if not traces:
        print(f"{label}: profile 失败；{output_dir} 中没有非空 kernel trace CSV")
        profile_results[label] = False
        PROFILE_STATUS = "FAILED"
        return False
    profile_traces[label] = traces
    parsed = [parse_trace_csv(trace, benchmark_label) for trace in traces]
    passed = all(parsed)
    profile_results[label] = passed
    PROFILE_STATUS = "PASS" if passed else "FAILED"
    print(f"{label}: {'PROFILE PASS' if passed else 'PROFILE FAILED'}; parsed {sum(parsed)}/{len(traces)} trace CSV")
    return passed

coalesced_trace_ok = False
linecross32_trace_ok = False
if not rocprof_available:
    print("Chapter6 profiling 未完成：rocprofv3 工具或权限不可用；内置参考数据仅用于教学，不满足 Pass Criteria。")
elif not (coalesced_ok and linecross1_ok and linecross32_ok):
    PROFILE_STATUS = "BLOCKED"
    print("Chapter6 profiling 未完成：3 个 benchmark 必须先全部成功；不使用旧 trace。")
else:
    coalesced_trace_ok = run_profile(
        "coalesced", "coalesced_size16777216",
        ["--kernel", "coalesced", "--size", "16777216", "--block", "256", "--warmup", "5", "--repeat", "10"],
    )
    linecross32_trace_ok = run_profile(
        "linecross_stride32", "linecross_stride32_size16777216",
        ["--kernel", "linecross", "--size", "16777216", "--block", "256", "--stride", "32", "--warmup", "5", "--repeat", "10"],
    )
    CH6_PROFILE_COMPLETE = coalesced_trace_ok and linecross32_trace_ok
    if not CH6_PROFILE_COMPLETE:
        print("Chapter6 profiling 未完成：coalesced 与 linecross_stride32 都必须成功采集并解析。内置参考数据仅用于教学，不满足 Pass Criteria。")

### kernel trace 结果解读

跳过最前面的 5 行 warmup，再统计后 10 行。参考运行得到：

| kernel | 单次最短时间 | 中位数 | Grid Size | VGPR | SGPR |
|--------|--------------|--------|-----------|------|------|
| `kernel_coalesced` | 329 μs | 330 μs | 16,777,216 | 8 | 128 |
| `kernel_linecross`（stride=32） | 2202 μs | 2304 μs | 524,288 | 16 | 128 |

这个表先读出三件事：

1. kernel trace 的 329 μs 和 benchmark 的 0.334 ms 基本一致，两种计时方法互相对得上；
2. `kernel_linecross` 是更慢的 dispatch；
3. 它的 Grid Size 只有连续访存版的 1/32，说明线程工作划分确实一起变了。

这就是 kernel trace 的第一价值：**先把“程序慢”缩小成“某个 kernel 慢”，再看这个 kernel 的启动配置。**

## 6.4 先列出一起变化的东西

这一节不增加新工具，只检查实验到底同时改了哪些变量。

从源码和 trace 可以列出：

| 变化 | coalesced | linecross stride=32 |
|------|-----------|---------------------|
| 同时访问的地址 | 相邻 | 更分散 |
| 每个线程处理的输出 | 1 个 | 32 个 |
| Grid Size | 16,777,216 | 524,288 |
| kernel 时间 | 0.334 ms | 2.25 ms |

*（图示：一次改了多件事时，先不要急着把结果归给其中一件。linecross 更慢可能因为地址排布变了、每线程循环次数变了、Grid Size 变了——它们都需要新的公平对照。）*

访存合并是一个合理方向，但并不是当前数据唯一支持的解释。有效带宽从 603 GB/s 降到 89.7 GB/s，仍只是同一份时间结果换成了带宽单位，不算第二份独立测量。

## 6.5 看看静态资源有没有变

这一节检查 VGPR、SGPR 和 LDS。Occupancy（占用率）在这里可以先简单理解成“GPU 能同时保留多少个 wavefront 轮流工作”。

`linecross stride=1` 和 `linecross stride=32` 执行的是同一个编译后的 kernel。stride 是运行时参数，因此两种配置的静态资源分配相同：

| 资源 | linecross stride=1 | linecross stride=32 |
|------|------------------:|-------------------:|
| VGPR | 16 | 16 |
| SGPR | 128 | 128 |
| LDS | 0 | 0 |

这说明寄存器和 LDS 分配不是两个 stride 配置之间的变量。不过，stride 仍然改变了每个线程的循环次数和 Grid Size，因此仍需要额外对照，才能判断剩余差距有多少来自访存合并。

这一节的结论很窄：**静态资源没变，但工作划分变了。**

### 步骤8：内置参考数据 — 加载已有证据

当 profiler 不可用时，加载仓库中已有的内置参考数据；这些数据仅供教学解读，不替代本机 profile。

In [ ]:
import json

# 内置参考数据：加载已有的实测数据
reference_data = {
    "platform": "Radeon RX 9070 XT (gfx1201) + ROCm 7.13",
    "measured": {
        "coalesced": {
            "min_time_ms": 0.334,
            "median_time_ms": 0.337,
            "effective_bw_gbps": 603,
            "grid_size": 16777216,
            "vgpr": 8,
            "sgpr": 128
        },
        "linecross_stride1": {
            "min_time_ms": 0.336,
            "median_time_ms": 0.338,
            "effective_bw_gbps": 599,
            "grid_size": 16777216,
            "vgpr": 16,
            "sgpr": 128
        },
        "linecross_stride32": {
            "min_time_ms": 2.25,
            "median_time_ms": 2.30,
            "effective_bw_gbps": 89.7,
            "grid_size": 524288,
            "vgpr": 16,
            "sgpr": 128
        }
    },
    "hypothesis": [
        "linecross stride=32 同时改变了地址排布、每线程循环次数和 Grid Size",
        "Grid Size 从 16M 降到 524K（1/32），说明线程工作划分确实变了",
        "VGPR/SGPR 分配相同（stride 是运行时参数），静态资源不是变量",
        "有效带宽从 603 GB/s 降到 89.7 GB/s（约 6.7 倍差距）"
    ]
}

print("=== 内置参考数据（9070XT + ROCm 7.13 的实测结果）===")
print(json.dumps(reference_data, indent=2, ensure_ascii=False))

## 6.6 用 stride 扫描观察趋势

这一节扫描 `stride`，观察这个 `linecross` 实现的整体性能怎样变化。

### 步骤9：stride 扫描（观察趋势）

扫描多个 stride 值，观察 linecross 实现的整体性能变化：

In [ ]:
stride_scan_status = "NOT_RUN"
stride_results = {}

def print_stride_reference():
    print("\n内置参考数据：stride 扫描结果（不是当前云端实测）")
    stride_reference = [
        (1, 0.338, 596),
        (8, 0.405, 497),
        (16, 1.65, 122),
        (32, 2.19, 92.1),
        (64, 4.86, 41.4),
        (256, 25.3, 7.95),
    ]
    print(f"{'stride':>6} | {'time_ms':>8} | {'eff_bw_gbps':>12} | {'vs_stride1':>10}")
    print("-" * 45)
    for s, t, bw in stride_reference:
        ratio = t / stride_reference[0][1]
        print(f"{s:>6} | {t:>8.3f} | {bw:>12.2f} | {ratio:>9.2f}x")

if not (RUN_READY and COMPILE_OK and COMPILE_TOKEN and output_binary.is_file()):
    stride_scan_status = "BLOCKED"
    print("stride 扫描 BLOCKED（本轮编译未成功，不使用旧 binary/JSON）")
    print_stride_reference()
else:
    stride_scan_status = "PASS"
    stride_dir = logs_dir / "stride" / f"{GPU_ARCH}_{COMPILE_TOKEN.replace(':', '_')}"
    stride_values = [1, 2, 4, 8, 16, 32, 64, 128, 256]
    print("stride 扫描:")
    for s in stride_values:
        output_json = stride_dir / f"linecross_s{s}.json"
        try:
            stride_dir.mkdir(parents=True, exist_ok=True)
            output_json.unlink(missing_ok=True)
        except OSError as exc:
            stride_scan_status = "FAILED"
            print(f"stride={s}: FAILED before launch; cannot isolate JSON: {exc}")
            break
        cmd = [
            str(output_binary),
            "--kernel", "linecross",
            "--size", "16777216",
            "--block", "256",
            "--stride", str(s),
            "--warmup", "20",
            "--repeat", "50",
            "--output-json", str(output_json),
        ]
        result = run_capture(cmd, timeout=300, cwd=WORK_DIR)
        if result is None or result.returncode != 0:
            returncode = "UNAVAILABLE" if result is None else result.returncode
            stderr = "<see command diagnostics>" if result is None else (result.stderr.strip() or "<empty>")
            print(f"stride={s}: FAILED returncode={returncode} stderr={stderr}")
            stride_scan_status = "FAILED"
            break
        print(result.stdout)
        for line in result.stdout.splitlines():
            if "stride" in line.lower() or "time" in line.lower() or "bandwidth" in line.lower():
                print(f"  {line}")
        if not output_json.is_file() or output_json.stat().st_size == 0:
            print(f"stride={s}: FAILED; fresh JSON missing")
            stride_scan_status = "FAILED"
            break
        try:
            stride_results[s] = json.loads(output_json.read_text(encoding="utf-8"))
        except (OSError, ValueError) as exc:
            print(f"stride={s}: FAILED; invalid fresh JSON: {exc}")
            stride_scan_status = "FAILED"
            break
    if stride_scan_status == "FAILED":
        stride_results = {}
        print("stride 扫描状态: FAILED（不消费任何旧 JSON）")
        print_stride_reference()
    else:
        print(f"stride 扫描状态: {stride_scan_status}; fresh results={len(stride_results)}")

### stride 扫描结果解读

参考值摘要：

| stride | 最短时间 | 有效带宽 | 相对 stride=1 耗时 |
|-------:|---------:|---------:|-------------------:|
| 1 | 0.338 ms | 596 GB/s | 1.00× |
| 8 | 0.405 ms | 497 GB/s | 1.20× |
| 16 | 1.65 ms | 122 GB/s | 4.89× |
| 32 | 2.19 ms | 92.1 GB/s | 6.47× |
| 64 | 4.86 ms | 41.4 GB/s | 14.4× |
| 256 | 25.3 ms | 7.95 GB/s | 74.9× |

可以直接观察到：stride 整体越大，这个实现越慢。但一个命令行参数同时改变了地址跨度、每线程循环次数和 Grid Size，所以这条曲线描述的是**组合效果**，不是单独的 cache line 或合并访存曲线。

### 公平对照应该怎么设计

要单独验证访存合并，下一组实验需要固定三件事：

1. 启动相同数量的线程和 wavefront；
2. 每个线程执行相同次数的循环和加法；
3. 只改变循环里的索引公式，让一版地址相邻、另一版地址分散。

例如，两版都让每个 lane 处理 32 个元素，只改变访问顺序：

```text
连续版：i = tile_base + j * 32 + lane
分散版：i = tile_base + lane * 32 + j
```

这才是后续应该补跑的公平对照。在这组新数据产生之前，本章停在“找到慢 kernel，并发现实验同时改变了多个底层变量”这个结论上。

*（图示：本章走完的最小 profiling 路线——benchmark 确认差距 → kernel trace 找到慢 dispatch → 列出所有变化变量 → 设计公平对照 → 再决定优化方向。profiler 不会自动替你证明原因。它先帮你找到慢点；真正解释原因，还需要源码检查和公平对照。）*

In [ ]:
benchmark_complete = all((coalesced_ok, linecross1_ok, linecross32_ok))
CH6_PROFILE_COMPLETE = benchmark_complete and coalesced_trace_ok and linecross32_trace_ok
print(f"benchmark complete (3/3): {benchmark_complete}")
print(f"profile complete (2/2 parsed): {CH6_PROFILE_COMPLETE}")
print(f"stride scan (separate observation): {stride_scan_status}")
if not CH6_PROFILE_COMPLETE:
    print("Chapter6 profiling 未完成：内置参考数据仅用于教学，不满足 Pass Criteria。")

## Expected Output / Interpretation

### benchmark 预期输出

在 9070XT (gfx1201) + ROCm 7.13 上，预期看到：

| kernel | stride | 最短时间 | 有效带宽 | 正确性 |
|--------|--------|----------|----------|--------|
| coalesced | - | 0.334 ms | 603 GB/s | OK |
| linecross | 1 | 0.336 ms | 599 GB/s | OK |
| linecross | 32 | 2.25 ms | 89.7 GB/s | OK |

**解读**：
- `linecross stride=1` 与 coalesced 时间接近（每个线程也只处理一个元素）
- `linecross stride=32` 慢约 6.7 倍，但同时改变了地址排布、循环次数和 Grid Size

### kernel trace 预期输出（如果 rocprofv3 可用）

CSV 文件中关键列：

| kernel | 单次最短时间 | 中位数 | Grid Size | VGPR | SGPR |
|--------|--------------|--------|-----------|------|------|
| kernel_coalesced | 329 μs | 330 μs | 16,777,216 | 8 | 128 |
| kernel_linecross (s=32) | 2202 μs | 2304 μs | 524,288 | 16 | 128 |

**解读**：
- kernel trace 的 329 μs 与 benchmark 的 0.334 ms 基本一致
- `kernel_linecross` 的 Grid Size 只有 coalesced 的 1/32
- VGPR/SGPR 分配相同（stride 是运行时参数）

### stride 扫描预期趋势

| stride | 相对 stride=1 耗时 |
|--------|-------------------|
| 1 | 1.00× |
| 8 | 1.20× |
| 16 | 4.89× |
| 32 | 6.47× |
| 64 | 14.4× |
| 256 | 74.9× |

**解读**：
- stride 整体越大，这个实现越慢
- 但一个参数同时改变了多个底层变量，这是**组合效果**
- 要单独验证访存合并，需要固定线程数和循环次数，只改索引公式

### 标签说明

- **measured**: 实测数据（来自 GPU event 或 rocprofv3）
- **内置参考数据**：profiler 不可用时使用的仓库数据，仅用于教学解读，不替代本机 profile
- **hypothesis**: 当前假设（需要后续公平对照验证）

## Pass Criteria

本章只有在 `CH6_PROFILE_COMPLETE=True` 时才算当前平台完成。它要求 **3 个 benchmark 成功**（coalesced、linecross stride=1、linecross stride=32）且 **2 个 profile 成功并解析**（coalesced、linecross stride=32）；stride 扫描单独报告，不计入完成条件。

### 必须满足

1. ✅ 使用带 arch 后缀的 `vector_add_bench_<gfx...>` 完成本轮 3/3 benchmark；后缀名与文档中的 `vector_add_bench` 是同一 benchmark，只是隔离不同架构的旧产物。
2. ✅ `rocprofv3` 可用时，采集并解析 coalesced 与 linecross stride=32 的每个非空 kernel trace CSV。
3. ✅ 对每个 kernel 行解读 Kernel_Name、Start/End、Grid、VGPR、SGPR（缺列为 `unavailable`），并报告 dispatch 数、最短/中位时间及与 benchmark 的对照。
4. ✅ 确认 `linecross stride=32` 明显慢于 coalesced，并理解 stride 同时改变地址排布、循环次数和 Grid Size。
5. ✅ `CH6_PROFILE_COMPLETE=True`；否则明确显示“Chapter6 profiling未完成”。

### 内置参考数据与 stride 扫描

- 工具、权限或 trace 失败时会显示“Chapter6 profiling 未完成”。内置参考数据仅供教学解读，**不满足 Pass Criteria**。
- stride 扫描只报告趋势和自身状态；它不替代 2 个 profile 成功/解析，也不决定 `CH6_PROFILE_COMPLETE`。

### 常见问题排查

| 现象 | 可能原因 | 解决方法 |
|------|----------|----------|
| hipcc 编译失败 | 云端预装运行时未提供编译器 | 联系云平台确认 hipcc/ROCm 可用性 |
| rocprofv3 不可用 | 工具或权限不可用 | 跳过 trace，使用内置参考数据（不替代本机 profile） |
| 时间差异很大 | 后台负载 | 关闭其他 GPU 任务 |
| CSV 文件为空 | profiler 权限问题 | 检查 `/tmp` 写权限 |
| 正确性检查失败 | 输入规模或 stride 不匹配 | 检查命令行参数 |

## 本章小结

- benchmark 先告诉你“哪个配置更慢”；`rocprofv3 --kernel-trace` 再告诉你“慢在哪个 dispatch”。
- `linecross stride=32` 约为 2.25 ms，明显慢于连续访存版的 0.334 ms。
- 当前 `linecross` 同时改变地址排布、每线程循环次数和 Grid Size，因此这组 6.7 倍差距不足以全部归因于访存合并。
- 下一步应固定线程数和每线程工作量，只改变索引公式，再重新测量。

---

## 延伸阅读

- [ROCprofiler 文档](https://rocm.docs.amd.com/projects/rocprofiler/en/latest/)
- [HIP Performance Guidelines](https://rocm.docs.amd.com/projects/HIP/en/latest/how-to/performance_guidelines.html)
- [GPUOpen：Memory Coalescing](https://gpuopen.com/learn/gcn-memory-coalescing/)

**下一章**: [第7章 读懂 Roofline 图](./chapter7.ipynb)